# Physics-term ablation under strict LOTO (Table 6)

Nine ablation conditions across four folds. The first cell covers M3, M4, and the single-term variants; the second cell adds the M2 row.


**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).

**Input dependency.** This notebook is a T4-row extension. It reads the existing 3-fold AUROC values from a CSV produced by an earlier run, computes the T4 row, and merges. Required input:

- `Processed_Data/NB10g_ablation_aggregate.csv`
- `Processed_Data/NB10d_term_ablation.csv`

If the file is missing, the notebook raises an explicit error.


In [ ]:
import os, glob, warnings, time
import numpy as np
import pandas as pd
import h5py
import scipy.stats as sst
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
np.random.seed(42)

ROOT = r"D:\Research\RCM"
BASE = os.path.join(ROOT, "Lab_Data")
OUT  = os.path.join(ROOT, "Processed_Data")

CANONICAL_CSV = os.path.join(OUT, "NB10g_ablation_aggregate.csv")
OUT_CSV       = os.path.join(OUT, "NB10g_ablation_aggregate_4fold.csv")

# Constants
TASK_PAYLOAD = {"T1": 0.0, "T2": 1.0, "T3": 3.0, "T4": 2.0}
PAYLOAD_COM  = np.array([0.0, 0.0, 0.05])
GRAVITY      = np.array([0.0, 0.0, -9.81])
RATE         = 125
SUBSAMPLE    = 4
MIN_SAMP     = 200
N_BOOT       = 1000
N_PCA        = 50

UR5_DH_A     = [0,        -0.42500, -0.39225, 0,       0,       0      ]
UR5_DH_D     = [0.089159,  0,        0,        0.10915, 0.09465, 0.0823 ]
UR5_DH_ALPHA = [np.pi/2,   0,        0,        np.pi/2, -np.pi/2, 0     ]
UR5_MASS     = [3.7000, 8.3930, 2.2750, 1.2190, 1.2190, 0.1879]
UR5_COM      = [
    [ 0.0,     -0.02561,  0.00193],
    [ 0.21250,  0.0,      0.11336],
    [ 0.11993,  0.0,      0.02650],
    [ 0.0,     -0.00180,  0.01634],
    [ 0.0,      0.00180,  0.01634],
    [ 0.0,      0.0,     -0.00116],
]

def dh_transform(a, d, alpha, theta):
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [0,   sa,    ca,     d   ],
        [0,   0,     0,      1   ]
    ])

def gravity_torque(q, payload_mass=0.0):
    T = [np.eye(4)]
    for i in range(6):
        T.append(T[-1] @ dh_transform(
            UR5_DH_A[i], UR5_DH_D[i], UR5_DH_ALPHA[i], q[i]))
    com_world = [(T[i+1] @ np.array([*UR5_COM[i], 1.0]))[:3] for i in range(6)]
    masses = list(UR5_MASS)
    if payload_mass > 0:
        masses.append(payload_mass)
        com_world.append((T[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
    tau_g = np.zeros(6)
    dq = 1e-6
    for i in range(6):
        qp = q.copy(); qp[i] += dq
        Tp = [np.eye(4)]
        for jj in range(6):
            Tp.append(Tp[-1] @ dh_transform(
                UR5_DH_A[jj], UR5_DH_D[jj], UR5_DH_ALPHA[jj], qp[jj]))
        for jj in range(len(masses)):
            cp = ((Tp[jj+1] @ np.array([*UR5_COM[jj], 1.0]))[:3]
                  if jj < 6
                  else (Tp[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
            tau_g[i] += masses[jj] * GRAVITY @ ((cp - com_world[jj]) / dq)
    return tau_g

# Ablation model definitions
PHI_FNS = {
    "bias_only":        lambda tg, qd, sg, qa: np.array([1.0]),
    "inertia_only":     lambda tg, qd, sg, qa: np.array([qa, 1.0]),
    "friction_only":    lambda tg, qd, sg, qa: np.array([qd, sg, 1.0]),
    "gravity_only":     lambda tg, qd, sg, qa: np.array([tg, 1.0]),
    "M3_grav_vel_fric": lambda tg, qd, sg, qa: np.array([tg, qd, sg, 1.0]),
    "M4_full":          lambda tg, qd, sg, qa: np.array([tg, qd, sg, qa, 1.0]),
}

HAS_GRAVITY = {
    "bias_only": False, "inertia_only": False, "friction_only": False,
    "gravity_only": True, "M3_grav_vel_fric": True, "M4_full": True,
}

ALL_CONDITIONS = [
    "no_physics",
    "no_physics_pca50",
    "bias_only",
    "inertia_only",
    "friction_only",
    "gravity_only",
    "M3_grav_vel_fric",
    "M4_full",
]

PSR_FEAT_COLS_50 = (
    [f"J{j}_{s}" for j in range(6)
     for s in ["resid_mean", "resid_std", "resid_rms", "resid_max",
               "resid_skew", "resid_kurtosis",
               "grav_resid_std", "grav_resid_rms"]]
    + ["total_resid_rms", "J1J2_resid_corr"]
)

# REGISTRY (T1/T2/T3 + T4)
REGISTRY = {
    "T1_healthy":    ("T1_PickPlace/Healthy",  "UR5_T1_healthy_180cyc_*.h5",
                      "T1", "healthy", "none", 0.0),
    "T2_healthy":    ("T2_Assembly/Healthy",   "UR5_T2_healthy_180cyc_*.h5",
                      "T2", "healthy", "none", 0.0),
    "T3_healthy":    ("T3_Palletize/Healthy",  "UR5_T3_healthy_183cyc_*.h5",
                      "T3", "healthy", "none", 0.0),
    "T1_A2_0.5kg":   ("T1_PickPlace/A2", "UR5_T1_A2_0.5kg_gripper_40cyc_*.h5",
                      "T1", "A2", "0.5kg", 0.5),
    "T1_A2_1kg":     ("T1_PickPlace/A2", "UR5_T1_A2_1kg_gripper_40cyc_*.h5",
                      "T1", "A2", "1kg",   1.0),
    "T1_A2_2kg":     ("T1_PickPlace/A2", "UR5_T1_A2_2kg_gripper_40cyc_*.h5",
                      "T1", "A2", "2kg",   2.0),
    "T1_A3_10wraps": ("T1_PickPlace/A3", "UR5_T1_A3_1band_40cyc_*.h5",
                      "T1", "A3", "10wraps", 0.0),
    "T1_A3_17wraps": ("T1_PickPlace/A3", "UR5_T1_A3_3bands_40cyc_*.h5",
                      "T1", "A3", "17wraps", 0.0),
    "T1_A5_20mm":    ("T1_PickPlace/A5", "UR5_T1_A5_20mm_40cyc_*.h5",
                      "T1", "A5", "20mm", 0.0),
    "T1_A5_50mm":    ("T1_PickPlace/A5", "UR5_T1_A5_50mm_40cyc_*.h5",
                      "T1", "A5", "50mm", 0.0),
    "T1_A5_100mm":   ("T1_PickPlace/A5", "UR5_T1_A5_100mm_40cyc_*.h5",
                      "T1", "A5", "100mm", 0.0),
    "T2_A2_1.5kg":   ("T2_Assembly/A2", "UR5_T2_A2_1.5kg_gripper_40cyc_*.h5",
                      "T2", "A2", "1.5kg", 1.5),
    "T2_A2_2kg":     ("T2_Assembly/A2", "UR5_T2_A2_2kg_gripper_40cyc_*.h5",
                      "T2", "A2", "2kg",   2.0),
    "T2_A2_3kg":     ("T2_Assembly/A2", "UR5_T2_A2_3kg_gripper_40cyc_*.h5",
                      "T2", "A2", "3kg",   3.0),
    "T2_A3_7duct":   ("T2_Assembly/A3", "UR5_T2_A3_light_duct_40cyc_*.h5",
                      "T2", "A3", "7duct", 0.0),
    "T2_A3_14duct":  ("T2_Assembly/A3",
                      "UR5_T2_A3_medium_duct_40cyc_*_225508.h5",
                      "T2", "A3", "14duct", 0.0),
    "T2_A5_20mm":    ("T2_Assembly/A5", "UR5_T2_A5_20mm_40cyc_*.h5",
                      "T2", "A5", "20mm", 0.0),
    "T2_A5_50mm":    ("T2_Assembly/A5", "UR5_T2_A5_50mm_40cyc_*.h5",
                      "T2", "A5", "50mm", 0.0),
    "T2_A5_100mm":   ("T2_Assembly/A5", "UR5_T2_A5_100mm_40cyc_*.h5",
                      "T2", "A5", "100mm", 0.0),
    "T3_A2_3.5kg":   ("T3_Palletize/A2", "UR5_T3_A2_3.5kg_gripper_33cyc_*.h5",
                      "T3", "A2", "3.5kg", 3.5),
    "T3_A2_4kg":     ("T3_Palletize/A2", "UR5_T3_A2_4kg_gripper_33cyc_*.h5",
                      "T3", "A2", "4kg",   4.0),
    "T3_A2_5kg":     ("T3_Palletize/A2", "UR5_T3_A2_4.5kg_gripper_33cyc_*.h5",
                      "T3", "A2", "5kg",   5.0),
    "T3_A3_7duct":   ("T3_Palletize/A3",
                      "UR5_T3_A3_light_duct_33cyc_*_222457.h5",
                      "T3", "A3", "7duct", 0.0),
    "T3_A3_14duct":  ("T3_Palletize/A3",
                      "UR5_T3_A3_medium_duct_33cyc_*_205648.h5",
                      "T3", "A3", "14duct", 0.0),
    "T3_A5_20mm":    ("T3_Palletize/A5", "UR5_T3_A5_20mm_33cyc_*_172334.h5",
                      "T3", "A5", "20mm", 0.0),
    "T3_A5_50mm":    ("T3_Palletize/A5", "UR5_T3_A5_50mm_33cyc_*_164447.h5",
                      "T3", "A5", "50mm", 0.0),
    "T3_A5_100mm":   ("T3_Palletize/A5", "UR5_T3_A5_100mm_33cyc_*_160716.h5",
                      "T3", "A5", "100mm", 0.0),
    # T4 entries
    "T4_healthy":    ("T4_BinReorient/healthy",
                      "UR5_T4_healthy_session2_35cyc_*.h5",
                      "T4", "healthy", "none", 0.0),
    "T4_A2_0.5kg":   ("T4_BinReorient/anomaly", "UR5_T4_A2_0.5kg_35cyc_*.h5",
                      "T4", "A2", "0.5kg", 0.5),
    "T4_A2_1kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_1kg_35cyc_*.h5",
                      "T4", "A2", "1kg",   1.0),
    "T4_A2_2kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_2kg_35cyc_*.h5",
                      "T4", "A2", "2kg",   2.0),
    "T4_A3_7duct":   ("T4_BinReorient/anomaly", "UR5_T4_A3_7wraps_35cyc_*.h5",
                      "T4", "A3", "7duct", 0.0),
    "T4_A3_14duct":  ("T4_BinReorient/anomaly", "UR5_T4_A3_14wraps_35cyc_*.h5",
                      "T4", "A3", "14duct", 0.0),
    "T4_A5_20mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_20mm_35cyc_*.h5",
                      "T4", "A5", "20mm", 0.0),
    "T4_A5_50mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_50mm_35cyc_*.h5",
                      "T4", "A5", "50mm", 0.0),
    "T4_A5_100mm":   ("T4_BinReorient/anomaly", "UR5_T4_A5_100mm_35cyc_*.h5",
                      "T4", "A5", "100mm", 0.0),
}

def bootstrap_auroc_bca(y_true, y_score, n_boot=N_BOOT, seed=42):
    """BCa bootstrap 95% CI for AUROC. Identical to NB10g."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    auroc_obs = roc_auc_score(y_true, y_score)
    boot = np.zeros(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]; ys = y_score[idx]
        boot[b] = roc_auc_score(yt, ys) if 0 < yt.sum() < n else auroc_obs
    prop = np.clip(np.mean(boot < auroc_obs), 1e-6, 1 - 1e-6)
    z0   = sst.norm.ppf(prop)
    jack = np.zeros(n)
    for i in range(n):
        idx_j = np.concatenate([np.arange(i), np.arange(i + 1, n)])
        yt_j = y_true[idx_j]; ys_j = y_score[idx_j]
        jack[i] = (roc_auc_score(yt_j, ys_j)
                   if 0 < yt_j.sum() < len(yt_j) else auroc_obs)
    jm  = jack.mean()
    num = np.sum((jm - jack) ** 3)
    den = 6.0 * (np.sum((jm - jack) ** 2) ** 1.5)
    a   = num / den if den != 0 else 0.0
    ci  = {}
    for label, za in [("lo", sst.norm.ppf(0.025)), ("hi", sst.norm.ppf(0.975))]:
        p = sst.norm.cdf(z0 + (z0 + za) / (1 - a * (z0 + za)))
        ci[label] = float(np.quantile(boot, np.clip(p, 0.001, 0.999)))
    return float(auroc_obs), ci["lo"], ci["hi"]

def zscore_scores(Xtr, Xte):
    mu = Xtr.mean(0); sg = Xtr.std(0) + 1e-8
    return np.abs((Xte - mu) / sg).mean(1)

def extract_raw_features(cycle):
    """104-dim raw features."""
    cur = cycle["current"]
    f = {}
    for j in range(6):
        s = cur[:, j]
        N = len(s)
        rms = np.sqrt(np.mean(s**2))
        peak = np.abs(s).max()
        f[f"J{j}_mean"]      = s.mean()
        f[f"J{j}_std"]       = s.std()
        f[f"J{j}_min"]       = s.min()
        f[f"J{j}_max"]       = s.max()
        f[f"J{j}_range"]     = s.max() - s.min()
        f[f"J{j}_rms"]       = rms
        f[f"J{j}_skew"]      = float(sst.skew(s))
        f[f"J{j}_kurtosis"]  = float(sst.kurtosis(s))
        f[f"J{j}_mad"]       = float(np.mean(np.abs(s - s.mean())))
        f[f"J{j}_peak2peak"] = peak * 2.0
        f[f"J{j}_crest"]     = float(peak / rms) if rms > 0 else 0.0
        zcr = float(np.sum(np.diff(np.sign(s - s.mean())) != 0)) / N
        f[f"J{j}_zcr"]       = zcr
        f[f"J{j}_energy"]    = float(np.sum(s**2))
        freqs = np.fft.rfftfreq(N, d=1.0/RATE)
        mag   = np.abs(np.fft.rfft(s - s.mean()))
        if mag.sum() > 0:
            f[f"J{j}_dom_fft_freq"]      = float(freqs[np.argmax(mag)])
            f[f"J{j}_dom_fft_mag"]       = float(mag.max())
            f[f"J{j}_spectral_centroid"] = float((freqs * mag).sum() / mag.sum())
            sc = (freqs * mag).sum() / mag.sum()
            f[f"J{j}_spectral_bandwidth"] = float(
                np.sqrt(((freqs - sc)**2 * mag).sum() / mag.sum()))
        else:
            f[f"J{j}_dom_fft_freq"]       = 0.0
            f[f"J{j}_dom_fft_mag"]        = 0.0
            f[f"J{j}_spectral_centroid"]  = 0.0
            f[f"J{j}_spectral_bandwidth"] = 0.0
    # 2 cross-joint correlations
    f["corr_J1_J2"] = float(np.corrcoef(cur[:, 1], cur[:, 2])[0, 1])
    f["corr_J2_J3"] = float(np.corrcoef(cur[:, 2], cur[:, 3])[0, 1])
    return f

RAW_FEAT_COLS = None  # filled in after we extract one cycle

def fit_psr_weights(cycles, phi_fn):
    Phi = {j: [] for j in range(6)}
    I   = {j: [] for j in range(6)}
    for cyc in cycles:
        qd_a  = cyc["qd"]
        cur   = cyc["current"]
        tau_g = cyc["tau_g_cached"]
        qdd   = cyc["qdd_cached"]
        for ti, t in enumerate(cyc["sub_idx"]):
            for j in range(6):
                phi_j = phi_fn(tau_g[ti, j], qd_a[t, j],
                               np.sign(qd_a[t, j]), qdd[ti, j])
                Phi[j].append(phi_j)
                I[j].append(cur[t, j])
    weights = {}
    for j in range(6):
        w, _, _, _ = np.linalg.lstsq(
            np.array(Phi[j]), np.array(I[j]), rcond=None)
        weights[j] = w
    return weights

def extract_psr_features(cycles, phi_fn, weights, has_gravity):
    rows = []
    for cyc in cycles:
        qd_a  = cyc["qd"]
        cur   = cyc["current"]
        tau_g = cyc["tau_g_cached"]
        qdd   = cyc["qdd_cached"]
        n_sub = len(cyc["sub_idx"])
        res   = np.zeros((n_sub, 6))
        gr    = np.zeros((n_sub, 6))
        for ti, t in enumerate(cyc["sub_idx"]):
            for j in range(6):
                phi_j = phi_fn(tau_g[ti, j], qd_a[t, j],
                               np.sign(qd_a[t, j]), qdd[ti, j])
                res[ti, j] = cur[t, j] - phi_j @ weights[j]
                if has_gravity:
                    w = weights[j]
                    gr[ti, j] = cur[t, j] - (w[0] * tau_g[ti, j] + w[-1])
                else:
                    gr[ti, j] = res[ti, j]
        f = {}
        for j in range(6):
            r = res[:, j]; g = gr[:, j]
            f[f"J{j}_resid_mean"]      = r.mean()
            f[f"J{j}_resid_std"]       = r.std()
            f[f"J{j}_resid_rms"]       = np.sqrt(np.mean(r**2))
            f[f"J{j}_resid_max"]       = np.abs(r).max()
            f[f"J{j}_resid_skew"]      = float(sst.skew(r))
            f[f"J{j}_resid_kurtosis"]  = float(sst.kurtosis(r))
            f[f"J{j}_grav_resid_std"]  = g.std()
            f[f"J{j}_grav_resid_rms"]  = np.sqrt(np.mean(g**2))
        f["total_resid_rms"] = np.sqrt(np.mean(res**2))
        f["J1J2_resid_corr"] = float(
            np.corrcoef(res[:, 1], res[:, 2])[0, 1] if len(res) > 2 else 0.0)
        rows.append(f)
    df = pd.DataFrame(rows)
    return df

# Main pipeline
print("=" * 70)
print("NB10g_T4_extension -- compute T4 column for Table 4 ablation")
print("=" * 70)

# Step 1
print(f"\n[Step 1] Loading {os.path.basename(CANONICAL_CSV)}...")
if not os.path.exists(CANONICAL_CSV):
    raise RuntimeError(
        f"Canonical ablation CSV not found at {CANONICAL_CSV}. "
        "This is the source of the published Table 4 T1/T2/T3 values."
    )
canon = pd.read_csv(CANONICAL_CSV)
print(f"  Loaded {len(canon)} canonical rows.")
print(f"  Tasks: {sorted(canon['test_task'].unique())}")
print(f"  Conditions: {sorted(canon['condition'].unique())}")

# Step 2 — load all cycles
print("\n[Step 2] Loading HDF5 data...")
all_cycles = []
for key, (subdir, pattern, task, anomaly, severity, _) in REGISTRY.items():
    matches = sorted(glob.glob(os.path.join(BASE, subdir, pattern)))
    if not matches:
        print(f"  WARNING — not found: {key}")
        continue
    with h5py.File(matches[0], "r") as f:
        cnum    = f["cycle_number"][:].astype(int).ravel()
        q_all   = f["actual_q"][:]
        qd_all  = f["actual_qd"][:]
        cur_all = f["actual_current"][:]
    is_anom = 0 if anomaly == "healthy" else 1
    for c in np.unique(cnum[cnum > 0]):
        mask = cnum == c
        if mask.sum() >= MIN_SAMP:
            all_cycles.append({
                "q": q_all[mask], "qd": qd_all[mask],
                "current": cur_all[mask],
                "task": task, "anomaly": anomaly,
                "severity": severity, "is_anomaly": is_anom,
            })

healthy_cycles = [c for c in all_cycles if c["is_anomaly"] == 0]
print(f"  Total cycles: {len(all_cycles)} | Healthy: {len(healthy_cycles)}")
for t in ["T1", "T2", "T3", "T4"]:
    nh = sum(1 for c in healthy_cycles if c["task"] == t)
    na = sum(1 for c in all_cycles     if c["task"] == t and c["is_anomaly"] == 1)
    print(f"    {t}: {nh} healthy, {na} anomaly")

# Step 3 — precompute gravity torques and qdd
print("\n[Step 3] Precomputing gravity torques (one-time)...")
t0 = time.perf_counter()
for ci, cyc in enumerate(all_cycles):
    payload = TASK_PAYLOAD[cyc["task"]]
    q_a  = cyc["q"]; qd_a = cyc["qd"]
    N    = len(q_a)
    idx  = list(range(0, N, SUBSAMPLE))
    n_sub = len(idx)
    tau_g_arr = np.zeros((n_sub, 6))
    qdd_arr   = np.zeros((n_sub, 6))
    for ti, t in enumerate(idx):
        tau_g_arr[ti] = gravity_torque(q_a[t], payload_mass=payload)
        for j in range(6):
            qdd_arr[ti, j] = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                              if 0 < t < N - 1 else 0.0)
    cyc["tau_g_cached"] = tau_g_arr
    cyc["qdd_cached"]   = qdd_arr
    cyc["sub_idx"]      = idx
    if (ci + 1) % 200 == 0:
        print(f"  {ci+1}/{len(all_cycles)} cycles done...")
print(f"  Gravity torque precomputation: {time.perf_counter()-t0:.1f}s")

# Step 4 — precompute raw 104-dim features
print("\n[Step 4] Extracting raw 104-dim features...")
for cyc in all_cycles:
    cyc["raw_features"] = extract_raw_features(cyc)
RAW_FEAT_COLS = sorted(all_cycles[0]["raw_features"].keys())
print(f"  Raw feature dimension: {len(RAW_FEAT_COLS)}")

# Step 5 — Run T4 fold for each condition
print("\n[Step 5] Running T4 fold across all 8 conditions...")
test_task = "T4"
tr_tasks  = ["T1", "T2", "T3"]
tr_cycs   = [c for c in all_cycles if c["task"] in tr_tasks]
te_cycs   = [c for c in all_cycles if c["task"] == test_task]
tr_healthy = [c for c in tr_cycs if c["is_anomaly"] == 0]
print(f"  Training healthy: {len(tr_healthy)} cycles")
print(f"  Test set: {len(te_cycs)} cycles "
      f"({sum(1 for c in te_cycs if c['is_anomaly']==0)} healthy, "
      f"{sum(1 for c in te_cycs if c['is_anomaly']==1)} anomaly)")

t4_rows = []
for condition in ALL_CONDITIONS:
    print(f"\n  Condition: {condition}")
    t_cond = time.perf_counter()

    if condition == "no_physics":
        Xtr = np.array([[c["raw_features"][k] for k in RAW_FEAT_COLS]
                        for c in tr_healthy])
        X_te_all = np.array([[c["raw_features"][k] for k in RAW_FEAT_COLS]
                             for c in te_cycs])
        y_te = np.array([c["is_anomaly"] for c in te_cycs])
        feat_label = "raw104"

    elif condition == "no_physics_pca50":
        Xtr_raw = np.array([[c["raw_features"][k] for k in RAW_FEAT_COLS]
                            for c in tr_healthy])
        X_te_raw = np.array([[c["raw_features"][k] for k in RAW_FEAT_COLS]
                             for c in te_cycs])
        scaler  = StandardScaler().fit(Xtr_raw)
        pca     = PCA(n_components=N_PCA, random_state=42)
        pca.fit(scaler.transform(Xtr_raw))
        Xtr = pca.transform(scaler.transform(Xtr_raw))
        X_te_all = pca.transform(scaler.transform(X_te_raw))
        y_te = np.array([c["is_anomaly"] for c in te_cycs])
        feat_label = "pca50"

    else:
        phi_fn   = PHI_FNS[condition]
        has_grav = HAS_GRAVITY[condition]
        weights  = fit_psr_weights(tr_healthy, phi_fn)
        df_te = extract_psr_features(te_cycs, phi_fn, weights, has_grav)
        df_tr = extract_psr_features(tr_healthy, phi_fn, weights, has_grav)
        Xtr = df_tr[PSR_FEAT_COLS_50].values
        X_te_all = df_te[PSR_FEAT_COLS_50].values
        y_te = np.array([c["is_anomaly"] for c in te_cycs])
        feat_label = "psr50"

    scores = zscore_scores(Xtr, X_te_all)
    auroc, lo, hi = bootstrap_auroc_bca(y_te, scores)
    t4_rows.append({
        "condition": condition,
        "test_task": "T4",
        "feat_type": feat_label,
        "auroc":     round(auroc, 4),
        "ci_lo":     round(lo, 4),
        "ci_hi":     round(hi, 4),
        "n_healthy": int((y_te == 0).sum()),
        "n_anomaly": int((y_te == 1).sum()),
    })
    print(f"    AUROC = {auroc:.4f} [{lo:.4f}, {hi:.4f}]  "
          f"({time.perf_counter()-t_cond:.1f}s)")

# Step 6
print("\n[Step 6] Merging with canonical CSV...")
combined = pd.concat([canon, pd.DataFrame(t4_rows)], ignore_index=True)
combined.to_csv(OUT_CSV, index=False)
print(f"  Saved: {OUT_CSV} ({len(combined)} rows)")

# Step 7
print("\n[Integrity check] T1/T2/T3 ablation rows preserved:")
match = True
for _, canon_row in canon.iterrows():
    sel = combined[(combined["test_task"] == canon_row["test_task"]) &
                   (combined["condition"] == canon_row["condition"])]
    if len(sel) != 1 or float(sel.iloc[0]["auroc"]) != float(canon_row["auroc"]):
        match = False
        print(f"  DRIFT: {canon_row['test_task']} {canon_row['condition']}: "
              f"canonical={canon_row['auroc']} new={sel.iloc[0]['auroc']}")
if match:
    print(f"  All {len(canon)} canonical rows preserved exactly.")

# Step 8 — Print Table 4 (revised, 4-fold)
print("\n" + "=" * 100)
print("REVISED TABLE 4 (Physics term ablation, 4-fold LOTO)")
print("=" * 100)
print(f"  {'Condition':<28}{'T1 AUROC [CI]':<24}{'T2 AUROC [CI]':<24}"
      f"{'T3 AUROC [CI]':<24}{'T4 AUROC [CI]':<24}{'Mean':<8}")
print("  " + "-" * 130)

DISPLAY_LABELS = {
    "no_physics":        "No physics (raw features)",
    "no_physics_pca50":  "No physics, PCA-50",
    "bias_only":         "Bias only (offset)",
    "inertia_only":      "Inertia term",
    "friction_only":     "Friction term",
    "gravity_only":      "Gravity term",
    "M3_grav_vel_fric":  "M3: G+Vel+F+B (key ablation)",
    "M4_full":           "M4: G+Vel+F+I+B (proposed)",
}

for cond in ALL_CONDITIONS:
    label = DISPLAY_LABELS.get(cond, cond)
    row = f"  {label:<28}"
    aurocs = []
    for task in ["T1", "T2", "T3", "T4"]:
        sub = combined[(combined["test_task"] == task) &
                       (combined["condition"] == cond)]
        if len(sub) == 0:
            row += f"{'—':<24}"
            continue
        a = sub.iloc[0]["auroc"]; lo = sub.iloc[0]["ci_lo"]; hi = sub.iloc[0]["ci_hi"]
        row += f"{a:.3f} [{lo:.3f}, {hi:.3f}]   "
        aurocs.append(a)
    mean_a = np.mean(aurocs) if aurocs else float("nan")
    row += f"{mean_a:.3f}"
    print(row)

print("=" * 100)

# Step 9
M2_PATH = os.path.join(OUT, "NB10d_term_ablation.csv")
if os.path.exists(M2_PATH):
    print("\n[Step 9] M2 row (G+Vel+B) — need T4 from NB10d_term_ablation_4fold.csv")
    m2_df = pd.read_csv(M2_PATH)
    m2_3fold = m2_df[m2_df["condition"] == "M2_gravity_vel"]
    if not m2_3fold.empty:
        print(f"  Canonical M2 (T1/T2/T3) values from NB10d_term_ablation.csv:")
        print(m2_3fold[["test_task", "auroc", "ci_lo", "ci_hi"]].to_string(index=False))
        print("  T4 M2 (G+Vel+B): NOT computed in this script. To get the full")
        print("  Table 4 9-row format, run NB10d Analysis 2 extended to T4.")
else:
    print(f"\n[Step 9] {M2_PATH} not found — M2 row will need separate extension.")

print("\nNB10g_T4_extension complete.")

In [ ]:
import os, glob, warnings, time
import numpy as np
import pandas as pd
import h5py
import scipy.stats as sst
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
np.random.seed(42)

ROOT = r"D:\Research\RCM"
BASE = os.path.join(ROOT, "Lab_Data")
OUT  = os.path.join(ROOT, "Processed_Data")

CANONICAL_CSV = os.path.join(OUT, "NB10d_term_ablation.csv")
OUT_CSV       = os.path.join(OUT, "NB10d_term_ablation_4fold.csv")

# Constants
TASK_PAYLOAD = {"T1": 0.0, "T2": 1.0, "T3": 3.0, "T4": 2.0}
PAYLOAD_COM  = np.array([0.0, 0.0, 0.05])
GRAVITY      = np.array([0.0, 0.0, -9.81])
RATE         = 125
SUBSAMPLE    = 4
MIN_SAMP     = 200
N_BOOT       = 10000

# M2 ablation model: Phi = [tau_g, qd, 1.0]
M2_PHI = lambda tg, qd, sg, qa: np.array([tg, qd, 1.0])
M2_HAS_GRAVITY = True
M2_NAME = "M2_gravity_vel"

UR5_DH_A     = [0,        -0.42500, -0.39225, 0,       0,       0      ]
UR5_DH_D     = [0.089159,  0,        0,        0.10915, 0.09465, 0.0823 ]
UR5_DH_ALPHA = [np.pi/2,   0,        0,        np.pi/2, -np.pi/2, 0     ]
UR5_MASS     = [3.7000, 8.3930, 2.2750, 1.2190, 1.2190, 0.1879]
UR5_COM      = [
    [ 0.0,     -0.02561,  0.00193],
    [ 0.21250,  0.0,      0.11336],
    [ 0.11993,  0.0,      0.02650],
    [ 0.0,     -0.00180,  0.01634],
    [ 0.0,      0.00180,  0.01634],
    [ 0.0,      0.0,     -0.00116],
]

def dh_transform(a, d, alpha, theta):
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [0,   sa,    ca,     d   ],
        [0,   0,     0,      1   ]
    ])

def gravity_torque(q, payload_mass=0.0):
    T = [np.eye(4)]
    for i in range(6):
        T.append(T[-1] @ dh_transform(
            UR5_DH_A[i], UR5_DH_D[i], UR5_DH_ALPHA[i], q[i]))
    com_world = [(T[i+1] @ np.array([*UR5_COM[i], 1.0]))[:3] for i in range(6)]
    masses = list(UR5_MASS)
    if payload_mass > 0:
        masses.append(payload_mass)
        com_world.append((T[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
    tau_g = np.zeros(6)
    dq = 1e-6
    for i in range(6):
        qp = q.copy(); qp[i] += dq
        Tp = [np.eye(4)]
        for jj in range(6):
            Tp.append(Tp[-1] @ dh_transform(
                UR5_DH_A[jj], UR5_DH_D[jj], UR5_DH_ALPHA[jj], qp[jj]))
        for jj in range(len(masses)):
            cp = ((Tp[jj+1] @ np.array([*UR5_COM[jj], 1.0]))[:3]
                  if jj < 6
                  else (Tp[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
            tau_g[i] += masses[jj] * GRAVITY @ ((cp - com_world[jj]) / dq)
    return tau_g

# REGISTRY (T1-T4)
REGISTRY = {
    "T1_healthy":    ("T1_PickPlace/Healthy",  "UR5_T1_healthy_180cyc_*.h5",
                      "T1", "healthy", "none", 0.0),
    "T2_healthy":    ("T2_Assembly/Healthy",   "UR5_T2_healthy_180cyc_*.h5",
                      "T2", "healthy", "none", 0.0),
    "T3_healthy":    ("T3_Palletize/Healthy",  "UR5_T3_healthy_183cyc_*.h5",
                      "T3", "healthy", "none", 0.0),
    "T1_A2_0.5kg":   ("T1_PickPlace/A2", "UR5_T1_A2_0.5kg_gripper_40cyc_*.h5",
                      "T1", "A2", "0.5kg", 0.5),
    "T1_A2_1kg":     ("T1_PickPlace/A2", "UR5_T1_A2_1kg_gripper_40cyc_*.h5",
                      "T1", "A2", "1kg",   1.0),
    "T1_A2_2kg":     ("T1_PickPlace/A2", "UR5_T1_A2_2kg_gripper_40cyc_*.h5",
                      "T1", "A2", "2kg",   2.0),
    "T1_A3_10wraps": ("T1_PickPlace/A3", "UR5_T1_A3_1band_40cyc_*.h5",
                      "T1", "A3", "10wraps", 0.0),
    "T1_A3_17wraps": ("T1_PickPlace/A3", "UR5_T1_A3_3bands_40cyc_*.h5",
                      "T1", "A3", "17wraps", 0.0),
    "T1_A5_20mm":    ("T1_PickPlace/A5", "UR5_T1_A5_20mm_40cyc_*.h5",
                      "T1", "A5", "20mm", 0.0),
    "T1_A5_50mm":    ("T1_PickPlace/A5", "UR5_T1_A5_50mm_40cyc_*.h5",
                      "T1", "A5", "50mm", 0.0),
    "T1_A5_100mm":   ("T1_PickPlace/A5", "UR5_T1_A5_100mm_40cyc_*.h5",
                      "T1", "A5", "100mm", 0.0),
    "T2_A2_1.5kg":   ("T2_Assembly/A2", "UR5_T2_A2_1.5kg_gripper_40cyc_*.h5",
                      "T2", "A2", "1.5kg", 1.5),
    "T2_A2_2kg":     ("T2_Assembly/A2", "UR5_T2_A2_2kg_gripper_40cyc_*.h5",
                      "T2", "A2", "2kg",   2.0),
    "T2_A2_3kg":     ("T2_Assembly/A2", "UR5_T2_A2_3kg_gripper_40cyc_*.h5",
                      "T2", "A2", "3kg",   3.0),
    "T2_A3_7duct":   ("T2_Assembly/A3", "UR5_T2_A3_light_duct_40cyc_*.h5",
                      "T2", "A3", "7duct", 0.0),
    "T2_A3_14duct":  ("T2_Assembly/A3", "UR5_T2_A3_medium_duct_40cyc_*_225508.h5",
                      "T2", "A3", "14duct", 0.0),
    "T2_A5_20mm":    ("T2_Assembly/A5", "UR5_T2_A5_20mm_40cyc_*.h5",
                      "T2", "A5", "20mm", 0.0),
    "T2_A5_50mm":    ("T2_Assembly/A5", "UR5_T2_A5_50mm_40cyc_*.h5",
                      "T2", "A5", "50mm", 0.0),
    "T2_A5_100mm":   ("T2_Assembly/A5", "UR5_T2_A5_100mm_40cyc_*.h5",
                      "T2", "A5", "100mm", 0.0),
    "T3_A2_3.5kg":   ("T3_Palletize/A2", "UR5_T3_A2_3.5kg_gripper_33cyc_*.h5",
                      "T3", "A2", "3.5kg", 3.5),
    "T3_A2_4kg":     ("T3_Palletize/A2", "UR5_T3_A2_4kg_gripper_33cyc_*.h5",
                      "T3", "A2", "4kg",   4.0),
    "T3_A2_5kg":     ("T3_Palletize/A2", "UR5_T3_A2_4.5kg_gripper_33cyc_*.h5",
                      "T3", "A2", "5kg",   5.0),
    "T3_A3_7duct":   ("T3_Palletize/A3", "UR5_T3_A3_light_duct_33cyc_*_222457.h5",
                      "T3", "A3", "7duct", 0.0),
    "T3_A3_14duct":  ("T3_Palletize/A3", "UR5_T3_A3_medium_duct_33cyc_*_205648.h5",
                      "T3", "A3", "14duct", 0.0),
    "T3_A5_20mm":    ("T3_Palletize/A5", "UR5_T3_A5_20mm_33cyc_*_172334.h5",
                      "T3", "A5", "20mm", 0.0),
    "T3_A5_50mm":    ("T3_Palletize/A5", "UR5_T3_A5_50mm_33cyc_*_164447.h5",
                      "T3", "A5", "50mm", 0.0),
    "T3_A5_100mm":   ("T3_Palletize/A5", "UR5_T3_A5_100mm_33cyc_*_160716.h5",
                      "T3", "A5", "100mm", 0.0),
    # T4 entries
    "T4_healthy":    ("T4_BinReorient/healthy", "UR5_T4_healthy_session2_35cyc_*.h5",
                      "T4", "healthy", "none", 0.0),
    "T4_A2_0.5kg":   ("T4_BinReorient/anomaly", "UR5_T4_A2_0.5kg_35cyc_*.h5",
                      "T4", "A2", "0.5kg", 0.5),
    "T4_A2_1kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_1kg_35cyc_*.h5",
                      "T4", "A2", "1kg",   1.0),
    "T4_A2_2kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_2kg_35cyc_*.h5",
                      "T4", "A2", "2kg",   2.0),
    "T4_A3_7duct":   ("T4_BinReorient/anomaly", "UR5_T4_A3_7wraps_35cyc_*.h5",
                      "T4", "A3", "7duct", 0.0),
    "T4_A3_14duct":  ("T4_BinReorient/anomaly", "UR5_T4_A3_14wraps_35cyc_*.h5",
                      "T4", "A3", "14duct", 0.0),
    "T4_A5_20mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_20mm_35cyc_*.h5",
                      "T4", "A5", "20mm", 0.0),
    "T4_A5_50mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_50mm_35cyc_*.h5",
                      "T4", "A5", "50mm", 0.0),
    "T4_A5_100mm":   ("T4_BinReorient/anomaly", "UR5_T4_A5_100mm_35cyc_*.h5",
                      "T4", "A5", "100mm", 0.0),
}

PSR_FEAT_COLS_50 = (
    [f"J{j}_{s}" for j in range(6)
     for s in ["resid_mean", "resid_std", "resid_rms", "resid_max",
               "resid_skew", "resid_kurtosis",
               "grav_resid_std", "grav_resid_rms"]]
    + ["total_resid_rms", "J1J2_resid_corr"]
)

def bootstrap_auroc_bca(y_true, y_score, n_boot=N_BOOT, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    auroc_obs = roc_auc_score(y_true, y_score)
    boot = np.zeros(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]; ys = y_score[idx]
        boot[b] = roc_auc_score(yt, ys) if 0 < yt.sum() < n else auroc_obs
    prop = np.clip(np.mean(boot < auroc_obs), 1e-6, 1 - 1e-6)
    z0   = sst.norm.ppf(prop)
    jack = np.zeros(n)
    for i in range(n):
        idx_j = np.concatenate([np.arange(i), np.arange(i + 1, n)])
        yt_j = y_true[idx_j]; ys_j = y_score[idx_j]
        jack[i] = (roc_auc_score(yt_j, ys_j)
                   if 0 < yt_j.sum() < len(yt_j) else auroc_obs)
    jm  = jack.mean()
    num = np.sum((jm - jack) ** 3)
    den = 6.0 * (np.sum((jm - jack) ** 2) ** 1.5)
    a   = num / den if den != 0 else 0.0
    ci  = {}
    for label, za in [("lo", sst.norm.ppf(0.025)), ("hi", sst.norm.ppf(0.975))]:
        p = sst.norm.cdf(z0 + (z0 + za) / (1 - a * (z0 + za)))
        ci[label] = float(np.quantile(boot, np.clip(p, 0.001, 0.999)))
    return float(auroc_obs), ci["lo"], ci["hi"]

def zscore_scores(Xtr, Xte):
    mu = Xtr.mean(0); sg = Xtr.std(0) + 1e-8
    return np.abs((Xte - mu) / sg).mean(1)

def fit_psr_weights(cycles, phi_fn):
    Phi = {j: [] for j in range(6)}
    I   = {j: [] for j in range(6)}
    for cyc in cycles:
        qd_a  = cyc["qd"]
        cur   = cyc["current"]
        tau_g = cyc["tau_g_cached"]
        qdd   = cyc["qdd_cached"]
        for ti, t in enumerate(cyc["sub_idx"]):
            for j in range(6):
                phi_j = phi_fn(tau_g[ti, j], qd_a[t, j],
                               np.sign(qd_a[t, j]), qdd[ti, j])
                Phi[j].append(phi_j)
                I[j].append(cur[t, j])
    weights = {}
    for j in range(6):
        w, _, _, _ = np.linalg.lstsq(
            np.array(Phi[j]), np.array(I[j]), rcond=None)
        weights[j] = w
    return weights

def extract_psr_features(cycles, phi_fn, weights, has_gravity):
    rows = []
    for cyc in cycles:
        qd_a  = cyc["qd"]
        cur   = cyc["current"]
        tau_g = cyc["tau_g_cached"]
        qdd   = cyc["qdd_cached"]
        n_sub = len(cyc["sub_idx"])
        res   = np.zeros((n_sub, 6))
        gr    = np.zeros((n_sub, 6))
        for ti, t in enumerate(cyc["sub_idx"]):
            for j in range(6):
                phi_j = phi_fn(tau_g[ti, j], qd_a[t, j],
                               np.sign(qd_a[t, j]), qdd[ti, j])
                res[ti, j] = cur[t, j] - phi_j @ weights[j]
                if has_gravity:
                    w = weights[j]
                    gr[ti, j] = cur[t, j] - (w[0] * tau_g[ti, j] + w[-1])
                else:
                    gr[ti, j] = res[ti, j]
        f = {}
        for j in range(6):
            r = res[:, j]; g = gr[:, j]
            f[f"J{j}_resid_mean"]     = r.mean()
            f[f"J{j}_resid_std"]      = r.std()
            f[f"J{j}_resid_rms"]      = np.sqrt(np.mean(r**2))
            f[f"J{j}_resid_max"]      = np.abs(r).max()
            f[f"J{j}_resid_skew"]     = float(sst.skew(r))
            f[f"J{j}_resid_kurtosis"] = float(sst.kurtosis(r))
            f[f"J{j}_grav_resid_std"] = g.std()
            f[f"J{j}_grav_resid_rms"] = np.sqrt(np.mean(g**2))
        f["total_resid_rms"] = np.sqrt(np.mean(res**2))
        f["J1J2_resid_corr"] = float(
            np.corrcoef(res[:, 1], res[:, 2])[0, 1] if len(res) > 2 else 0.0)
        rows.append(f)
    return pd.DataFrame(rows)

# Main
print("=" * 70)
print("NB10d_M2_T4_extension -- compute T4 row for M2 (G+Vel+B) ablation")
print("=" * 70)

print(f"\n[Step 1] Loading {os.path.basename(CANONICAL_CSV)}...")
if not os.path.exists(CANONICAL_CSV):
    raise RuntimeError(
        f"Canonical CSV not found at {CANONICAL_CSV}. "
        "This is the source of the published M2 (G+Vel+B) T1/T2/T3 values."
    )
canon = pd.read_csv(CANONICAL_CSV)
print(f"  Loaded {len(canon)} rows.")
print(f"  Columns: {list(canon.columns)}")
print(f"  Models: {sorted(canon['model'].unique())}")
m2_canon = canon[canon["model"] == M2_NAME]
print(f"  M2 rows in canonical: {len(m2_canon)}")
print(m2_canon[["model", "test_task", "train_tasks", "auroc",
                "ci_lo", "ci_hi"]].to_string(index=False))

# Step 2: load all cycles
print("\n[Step 2] Loading HDF5 data...")
all_cycles = []
for key, (subdir, pattern, task, anomaly, severity, _) in REGISTRY.items():
    matches = sorted(glob.glob(os.path.join(BASE, subdir, pattern)))
    if not matches:
        print(f"  WARNING — not found: {key}")
        continue
    with h5py.File(matches[0], "r") as f:
        cnum    = f["cycle_number"][:].astype(int).ravel()
        q_all   = f["actual_q"][:]
        qd_all  = f["actual_qd"][:]
        cur_all = f["actual_current"][:]
    is_anom = 0 if anomaly == "healthy" else 1
    for c in np.unique(cnum[cnum > 0]):
        mask = cnum == c
        if mask.sum() >= MIN_SAMP:
            all_cycles.append({
                "q": q_all[mask], "qd": qd_all[mask],
                "current": cur_all[mask],
                "task": task, "anomaly": anomaly,
                "severity": severity, "is_anomaly": is_anom,
            })
print(f"  Total cycles: {len(all_cycles)}")

# Step 3: precompute gravity torques (only for T4 fold cycles)
print("\n[Step 3] Precomputing gravity torques and qdd (T1+T2+T3+T4)...")
t0 = time.perf_counter()
for ci, cyc in enumerate(all_cycles):
    payload = TASK_PAYLOAD[cyc["task"]]
    q_a  = cyc["q"]; qd_a = cyc["qd"]
    N    = len(q_a)
    idx  = list(range(0, N, SUBSAMPLE))
    n_sub = len(idx)
    tau_g_arr = np.zeros((n_sub, 6))
    qdd_arr   = np.zeros((n_sub, 6))
    for ti, t in enumerate(idx):
        tau_g_arr[ti] = gravity_torque(q_a[t], payload_mass=payload)
        for j in range(6):
            qdd_arr[ti, j] = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                              if 0 < t < N - 1 else 0.0)
    cyc["tau_g_cached"] = tau_g_arr
    cyc["qdd_cached"]   = qdd_arr
    cyc["sub_idx"]      = idx
    if (ci + 1) % 200 == 0:
        print(f"  {ci+1}/{len(all_cycles)} cycles done...")
print(f"  Precomputation time: {time.perf_counter()-t0:.1f}s")

# Step 4: T4 LOTO fold for M2
print("\n[Step 4] M2 (G+Vel+B) on T4 fold (train: T1+T2+T3 healthy)...")
test_task = "T4"
tr_cycs = [c for c in all_cycles if c["task"] in ["T1", "T2", "T3"]]
te_cycs = [c for c in all_cycles if c["task"] == "T4"]
tr_healthy = [c for c in tr_cycs if c["is_anomaly"] == 0]
print(f"  Training healthy: {len(tr_healthy)} cycles")
print(f"  Test cycles: {len(te_cycs)} ({sum(1 for c in te_cycs if c['is_anomaly']==0)} healthy, "
      f"{sum(1 for c in te_cycs if c['is_anomaly']==1)} anomaly)")

t_cond = time.perf_counter()
weights = fit_psr_weights(tr_healthy, M2_PHI)
df_te = extract_psr_features(te_cycs, M2_PHI, weights, M2_HAS_GRAVITY)
df_tr = extract_psr_features(tr_healthy, M2_PHI, weights, M2_HAS_GRAVITY)
Xtr = df_tr[PSR_FEAT_COLS_50].values
X_te_all = df_te[PSR_FEAT_COLS_50].values
y_te = np.array([c["is_anomaly"] for c in te_cycs])
scores = zscore_scores(Xtr, X_te_all)
auroc, lo, hi = bootstrap_auroc_bca(y_te, scores)
print(f"  T4 M2 AUROC = {auroc:.4f} [{lo:.4f}, {hi:.4f}]  "
      f"({time.perf_counter()-t_cond:.1f}s)")

t4_m2_row = {
    "model":           M2_NAME,
    "n_terms":         3,
    "test_task":       "T4",
    "train_tasks":     "T1+T2+T3",
    "detector":        "ZScore",
    "n_healthy_test":  int((y_te == 0).sum()),
    "n_anomaly_test":  int((y_te == 1).sum()),
    "auroc":           round(auroc, 4),
    "ci_lo":           round(lo, 4),
    "ci_hi":           round(hi, 4),
    "ci_width":        round(hi - lo, 4),
}

# Step 5: merge and save
print("\n[Step 5] Merging with canonical CSV...")
combined = pd.concat([canon, pd.DataFrame([t4_m2_row])], ignore_index=True)
combined.to_csv(OUT_CSV, index=False)
print(f"  Saved: {OUT_CSV} ({len(combined)} rows)")

# Step 6: integrity check
print("\n[Integrity check] T1/T2/T3 rows preserved byte-for-byte:")
match = True
for _, canon_row in canon.iterrows():
    sel = combined[(combined["test_task"] == canon_row["test_task"]) &
                   (combined["model"] == canon_row["model"])]
    if len(sel) != 1 or float(sel.iloc[0]["auroc"]) != float(canon_row["auroc"]):
        match = False
        print(f"  DRIFT: {canon_row['test_task']} {canon_row['model']}: "
              f"canonical={canon_row['auroc']} new={sel.iloc[0]['auroc']}")
if match:
    print(f"  All {len(canon)} canonical rows preserved exactly.")

# Step 7: print the M2 row.
print("\n" + "=" * 75)
print("M2: G+Vel+B  --  Table 4 row (4 folds)")
print("=" * 75)
print(f"  {'Test fold':<10}{'AUROC [95% CI]':<28}")
print("  " + "-" * 60)
for task in ["T1", "T2", "T3", "T4"]:
    sub = combined[(combined["test_task"] == task) &
                   (combined["model"] == M2_NAME)]
    if len(sub) == 0:
        print(f"  {task:<10}{'—':<28}")
        continue
    a  = sub.iloc[0]["auroc"]
    lo = sub.iloc[0]["ci_lo"]
    hi = sub.iloc[0]["ci_hi"]
    print(f"  {task:<10}{a:.3f} [{lo:.3f}, {hi:.3f}]")
mean_m2 = combined[combined["model"] == M2_NAME]["auroc"].mean()
print(f"  {'Mean':<10}{mean_m2:.3f}")
print("=" * 75)
print("\nNB10d_M2_T4_extension complete.")